In [8]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [5]:
# Create all gold layer tables
DB_GOLD   = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")



tables = [
    "dim_channel",
    "dim_claim_type",
    "dim_member_segment",
    "dim_product_line",
    "dim_providers",
    "dim_region",
    "dm_claims_experience",
    "dm_member_value",
    "dm_policy_retention",
    "dq_monitoring",
    "fact_claims",
    "fact_members",
    "fact_policies",
    "ft_claims_risk",
    "ft_claims_risk_split",
    "ft_policy_churn",
    "ft_policy_churn_split",
    "ml_monitoring",
    "ml_monitoring_view",
    "scored_policy_churn",
    "star_claims",
    "star_members",
    "star_policies",
    "vw_claims_experience_kpi",
    "vw_member_profile",
    "vw_member_value_kpi",
    "vw_ml_claims_risk_features",
    "vw_ml_policy_churn_features",
    "vw_policy_portfolio",
    "vw_policy_retention_kpi",
    "vw_scored_claims_fraud",
    "vw_scored_claims_high_cost",
    "vw_scored_policy_churn"
]

for table in tables:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DB_GOLD}.{table}
        USING DELTA
        LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/{table}'
    """)

print(f"Created {len(tables)} tables in {DB_GOLD} database")

Created 33 tables in bupa_gold database


In [4]:
# Print schemas from all tables in the gold layer
for table_name in tables:
    try:
        df = spark.table(f"{DB_GOLD}.{table_name}")
        print(f"\n{'='*60}")
        print(f"Schema for {DB_GOLD}.{table_name}")
        print(f"{'='*60}")
        df.printSchema()
    except Exception as e:
        print(f"Error reading {DB_GOLD}.{table_name}: {str(e)}")


Schema for bupa_gold.dim_channel
root
 |-- Channel_Code: string (nullable = true)
 |-- Channel_Name: string (nullable = true)


Schema for bupa_gold.dim_claim_type
root
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Type: string (nullable = true)


Schema for bupa_gold.dim_member_segment
root
 |-- Member_Segment_Key: long (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Band: string (nullable = true)
 |-- Smoker_Status: string (nullable = true)
 |-- Chronic_Flag: integer (nullable = true)
 |-- Chronic_Group: string (nullable = true)
 |-- Employment_Group: string (nullable = true)
 |-- Region_Group: string (nullable = true)


Schema for bupa_gold.dim_product_line
root
 |-- Product_Line_Code: string (nullable = true)
 |-- Product_Line: string (nullable = true)


Schema for bupa_gold.dim_member_segment
root
 |-- Member_Segment_Key: long (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Band: string (nullable = true)
 |-- Smoker_Status: string

In [ ]:
import json

def dump_schema(df, path):
    schema = [
        {"name": f.name, "type": f.dataType.simpleString(), "nullable": f.nullable}
        for f in df.schema.fields
    ]
    with open(path, "w") as f:
        json.dump(schema, f, indent=2)

dump_schema(spark.table("bupa_gold.fact_claims"), "/Users/manojrammopati/Public/Projects/bupa_insurance_project/schemas/gold/fact_claims.json")
dump_schema(spark.table("bupa_gold.star_policies"), "schemas/gold/star_policies.json")
dump_schema(spark.table("bupa_gold.star_claims"), "schemas/gold/star_claims.json")
dump_schema(spark.table("bupa_gold.ft_policy_churn"), "schemas/gold/ft_policy_churn.json")
dump_schema(spark.table("bupa_gold.ft_claims_risk"), "schemas/gold/ft_claims_risk.json")
dump_schema(spark.table("bupa_gold.scored_policy_churn"), "schemas/gold/scored_policy_churn.json")


In [6]:
import json
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

DB = os.getenv("DB_GOLD", "bupa_gold")
BASE = os.getenv("LOCAL_GOLD_BASE", "data/gold")
OUT = Path("schemas/gold")
OUT.mkdir(parents=True, exist_ok=True)

TABLES = [
  "dim_channel","dim_claim_type","dim_member_segment","dim_product_line","dim_providers","dim_region",
  "fact_claims","fact_members","fact_policies",
  "dm_claims_experience","dm_member_value","dm_policy_retention",
  "star_claims","star_members","star_policies",
  "ft_policy_churn","ft_policy_churn_split","ft_claims_risk","ft_claims_risk_split",
  "dq_monitoring","ml_monitoring","ml_monitoring_view",
  "scored_policy_churn",
]

builder = (
    SparkSession.builder.master("local[*]").appName("schema-snapshot")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

for t in TABLES:
    path = f"{BASE}/{t}"
    df = spark.read.format("delta").load(path)
    schema_json = json.loads(df.schema.json())["fields"]
    with open(OUT / f"{t}.json", "w") as f:
        json.dump(schema_json, f, indent=2)
    print("Wrote", OUT / f"{t}.json")


25/12/15 16:39:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Wrote schemas/gold/dim_channel.json
Wrote schemas/gold/dim_claim_type.json
Wrote schemas/gold/dim_member_segment.json
Wrote schemas/gold/dim_member_segment.json
Wrote schemas/gold/dim_product_line.json
Wrote schemas/gold/dim_product_line.json
Wrote schemas/gold/dim_providers.json
Wrote schemas/gold/dim_region.json
Wrote schemas/gold/fact_claims.json
Wrote schemas/gold/dim_providers.json
Wrote schemas/gold/dim_region.json
Wrote schemas/gold/fact_claims.json
Wrote schemas/gold/fact_members.json
Wrote schemas/gold/fact_policies.json
Wrote schemas/gold/dm_claims_experience.json
Wrote schemas/gold/dm_member_value.json
Wrote schemas/gold/dm_policy_retention.json
Wrote schemas/gold/fact_members.json
Wrote schemas/gold/fact_policies.json
Wrote schemas/gold/dm_claims_experience.json
Wrote schemas/gold/dm_member_value.json
Wrote schemas/gold/dm_policy_retention.json
Wrote schemas/gold/star_claims.json
Wrote schemas/gold/star_members.json
Wrote schemas/gold/star_policies.json
Wrote schemas/gold/f

In [7]:
import json
import os
from pathlib import Path

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

BASE = os.getenv("LOCAL_GOLD_BASE", "data/gold")
OUT = Path("schemas/gold")
OUT.mkdir(parents=True, exist_ok=True)

TABLES = [
    "dim_channel","dim_claim_type","dim_member_segment","dim_product_line","dim_providers","dim_region",
    "dm_claims_experience","dm_member_value","dm_policy_retention",
    "dq_monitoring",
    "fact_claims","fact_members","fact_policies",
    "ft_claims_risk","ft_claims_risk_split","ft_policy_churn","ft_policy_churn_split",
    "ml_monitoring","ml_monitoring_view",
    "scored_policy_churn",
    "star_claims","star_members","star_policies",
    # “vw_*” folders exist locally → snapshot them too
    "vw_claims_experience_kpi","vw_member_profile","vw_member_value_kpi",
    "vw_ml_claims_risk_features","vw_ml_policy_churn_features",
    "vw_policy_portfolio","vw_policy_retention_kpi",
    "vw_scored_claims_fraud","vw_scored_claims_high_cost","vw_scored_policy_churn",
]

builder = (
    SparkSession.builder.master("local[*]").appName("schema-snapshot")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

for t in TABLES:
    path = f"{BASE}/{t}"
    df = spark.read.format("delta").load(path)
    fields = json.loads(df.schema.json())["fields"]
    out_path = OUT / f"{t}.json"
    out_path.write_text(json.dumps(fields, indent=2))
    print("Wrote", out_path)

spark.stop()


Wrote schemas/gold/dim_channel.json
Wrote schemas/gold/dim_claim_type.json
Wrote schemas/gold/dim_member_segment.json
Wrote schemas/gold/dim_product_line.json
Wrote schemas/gold/dim_providers.json
Wrote schemas/gold/dim_region.json
Wrote schemas/gold/dm_claims_experience.json
Wrote schemas/gold/dm_member_value.json
Wrote schemas/gold/dm_policy_retention.json
Wrote schemas/gold/dq_monitoring.json
Wrote schemas/gold/fact_claims.json
Wrote schemas/gold/fact_members.json
Wrote schemas/gold/fact_policies.json
Wrote schemas/gold/ft_claims_risk.json
Wrote schemas/gold/ft_claims_risk_split.json
Wrote schemas/gold/ft_policy_churn.json
Wrote schemas/gold/ft_policy_churn_split.json
Wrote schemas/gold/ml_monitoring.json
Wrote schemas/gold/ml_monitoring_view.json
Wrote schemas/gold/scored_policy_churn.json
Wrote schemas/gold/star_claims.json
Wrote schemas/gold/star_members.json
Wrote schemas/gold/star_policies.json
Wrote schemas/gold/star_policies.json
Wrote schemas/gold/vw_claims_experience_kpi.js

In [10]:
# Export sample data from all gold tables to local delta folder
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

BASE = os.getenv("LOCAL_GOLD_BASE", "data/gold")
SAMPLE_OUT = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample")
SAMPLE_OUT.mkdir(parents=True, exist_ok=True)

TABLES = [
    "dim_channel", "dim_claim_type", "dim_member_segment", "dim_product_line", "dim_providers", "dim_region",
    "fact_claims", "fact_members", "fact_policies",
    "dm_claims_experience", "dm_member_value", "dm_policy_retention",
    "star_claims", "star_members", "star_policies",
    "ft_policy_churn", "ft_policy_churn_split", "ft_claims_risk", "ft_claims_risk_split",
    "dq_monitoring", "ml_monitoring",
    "scored_policy_churn",
]

builder = (
    SparkSession.builder.master("local[*]").appName("export-gold-samples")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Exporting {len(TABLES)} sample tables to {SAMPLE_OUT}\n")

for t in TABLES:
    src_path = f"{BASE}/{t}"
    dst_path = str(SAMPLE_OUT / t)
    
    try:
        # Read full table
        df = spark.read.format("delta").load(src_path)
        total_rows = df.count()
        
        # Take sample (limit to first 1000 rows or 10% of data, whichever is smaller)
        sample_size = min(1000, max(10, int(total_rows * 0.1)))
        sample_df = df.limit(sample_size)
        
        # Write as Delta table
        sample_df.write.format("delta").mode("overwrite").save(dst_path)
        
        print(f"✅ {t:35s} - {total_rows:,} rows → {sample_size:,} samples exported")
        
    except Exception as e:
        print(f"⚠️ {t:35s} - Error: {str(e)[:60]}")

spark.stop()

print(f"\n{'='*70}")
print(f"✅ Samples exported to: {SAMPLE_OUT}")
print(f"{'='*70}")

Exporting 22 sample tables to /Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample

✅ dim_channel                         - 4 rows → 10 samples exported
✅ dim_channel                         - 4 rows → 10 samples exported


✅ dim_claim_type                      - 6 rows → 10 samples exported
✅ dim_member_segment                  - 1,440 rows → 144 samples exported
✅ dim_member_segment                  - 1,440 rows → 144 samples exported


✅ dim_product_line                    - 5 rows → 10 samples exported
✅ dim_providers                       - 5,393 rows → 539 samples exported
✅ dim_providers                       - 5,393 rows → 539 samples exported


✅ dim_region                          - 53 rows → 10 samples exported
✅ fact_claims                         - 558,211 rows → 1,000 samples exported
✅ fact_claims                         - 558,211 rows → 1,000 samples exported
✅ fact_members                        - 381,109 rows → 1,000 samples exported
✅ fact_members                        - 381,109 rows → 1,000 samples exported
✅ fact_policies                       - 381,109 rows → 1,000 samples exported
✅ fact_policies                       - 381,109 rows → 1,000 samples exported


✅ dm_claims_experience                - 558,211 rows → 1,000 samples exported
✅ dm_member_value                     - 381,109 rows → 1,000 samples exported
✅ dm_member_value                     - 381,109 rows → 1,000 samples exported
✅ dm_policy_retention                 - 183 rows → 18 samples exported
✅ dm_policy_retention                 - 183 rows → 18 samples exported
✅ star_claims                         - 558,211 rows → 1,000 samples exported
✅ star_claims                         - 558,211 rows → 1,000 samples exported
✅ star_members                        - 381,109 rows → 1,000 samples exported
✅ star_members                        - 381,109 rows → 1,000 samples exported
✅ star_policies                       - 381,109 rows → 1,000 samples exported
✅ star_policies                       - 381,109 rows → 1,000 samples exported
✅ ft_policy_churn                     - 381,109 rows → 1,000 samples exported
✅ ft_policy_churn                     - 381,109 rows → 1,000 samples exported


✅ ft_policy_churn_split               - 381,109 rows → 1,000 samples exported


✅ ft_claims_risk                      - 558,211 rows → 1,000 samples exported
✅ ft_claims_risk_split                - 558,211 rows → 1,000 samples exported
✅ ft_claims_risk_split                - 558,211 rows → 1,000 samples exported


✅ dq_monitoring                       - 24 rows → 10 samples exported
✅ ml_monitoring                       - 29 rows → 10 samples exported
✅ ml_monitoring                       - 29 rows → 10 samples exported
✅ scored_policy_churn                 - 314,128 rows → 1,000 samples exported
✅ scored_policy_churn                 - 314,128 rows → 1,000 samples exported

✅ Samples exported to: /Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample

✅ Samples exported to: /Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample


In [12]:
# Check source data before exporting samples
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

BASE = os.getenv("LOCAL_GOLD_BASE", "data/gold")

TABLES = [
    "dim_channel", "dim_claim_type", "dim_member_segment", "dim_product_line", "dim_providers", "dim_region",
    "fact_claims", "fact_members", "fact_policies",
    "dm_claims_experience", "dm_member_value", "dm_policy_retention",
    "star_claims", "star_members", "star_policies",
    "ft_policy_churn", "ft_policy_churn_split", "ft_claims_risk", "ft_claims_risk_split",
    "dq_monitoring", "ml_monitoring",
    "scored_policy_churn",
]

builder = (
    SparkSession.builder.master("local[*]").appName("check-gold-data")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Checking source data in {BASE}\n")
print(f"{'Table':<35s} {'Rows':>10s} {'Status':<20s}")
print(f"{'-'*65}")

for t in TABLES:
    src_path = f"{BASE}/{t}"
    
    try:
        # Check if path exists
        if not Path(src_path).exists():
            print(f"{t:<35s} {'N/A':>10s} {'❌ Path not found':<20s}")
            continue
            
        df = spark.read.format("delta").load(src_path)
        row_count = df.count()
        
        if row_count == 0:
            print(f"{t:<35s} {0:>10,d} {'⚠️ Empty table':<20s}")
        else:
            print(f"{t:<35s} {row_count:>10,d} {'✅ Has data':<20s}")
        
    except Exception as e:
        print(f"{t:<35s} {'N/A':>10s} {'❌ ' + str(e)[:15]:<20s}")

spark.stop()

Checking source data in data/gold

Table                                     Rows Status              
-----------------------------------------------------------------


dim_channel                                  4 ✅ Has data          
dim_claim_type                               6 ✅ Has data          
dim_member_segment                       1,440 ✅ Has data          


dim_product_line                             5 ✅ Has data          


dim_providers                            5,393 ✅ Has data          
dim_region                                  53 ✅ Has data          
dim_region                                  53 ✅ Has data          


fact_claims                            558,211 ✅ Has data          
fact_members                           381,109 ✅ Has data          
fact_members                           381,109 ✅ Has data          
fact_policies                          381,109 ✅ Has data          
fact_policies                          381,109 ✅ Has data          


dm_claims_experience                   558,211 ✅ Has data          
dm_member_value                        381,109 ✅ Has data          
dm_member_value                        381,109 ✅ Has data          
dm_policy_retention                        183 ✅ Has data          
dm_policy_retention                        183 ✅ Has data          
star_claims                            558,211 ✅ Has data          
star_claims                            558,211 ✅ Has data          


star_members                           381,109 ✅ Has data          
star_policies                          381,109 ✅ Has data          
star_policies                          381,109 ✅ Has data          


ft_policy_churn                        381,109 ✅ Has data          
ft_policy_churn_split                  381,109 ✅ Has data          
ft_policy_churn_split                  381,109 ✅ Has data          
ft_claims_risk                         558,211 ✅ Has data          
ft_claims_risk                         558,211 ✅ Has data          
ft_claims_risk_split                   558,211 ✅ Has data          
ft_claims_risk_split                   558,211 ✅ Has data          
dq_monitoring                               24 ✅ Has data          
dq_monitoring                               24 ✅ Has data          
ml_monitoring                               29 ✅ Has data          
ml_monitoring                               29 ✅ Has data          
scored_policy_churn                    314,128 ✅ Has data          
scored_policy_churn                    314,128 ✅ Has data          


In [14]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pandas as pd

SAMPLE_PATH = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample")

# Create Spark session
builder = (
    SparkSession.builder.master("local[*]").appName("load-gold-samples")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Get all table directories
table_dirs = sorted([d for d in SAMPLE_PATH.iterdir() if d.is_dir() and not d.name.startswith(".")])

print(f"\n{'='*80}")
print(f"Loading {len(table_dirs)} sample tables from {SAMPLE_PATH}")
print(f"{'='*80}\n")

# Store all data in a dictionary
all_data = {}

for table_dir in table_dirs:
    table_name = table_dir.name
    try:
        df = spark.read.format("delta").load(str(table_dir))
        row_count = df.count()
        col_count = len(df.columns)
        
        # Convert to pandas for easier display
        pdf = df.toPandas()
        all_data[table_name] = {
            "spark_df": df,
            "pandas_df": pdf,
            "rows": row_count,
            "cols": col_count,
            "schema": df.schema
        }
        
        print(f"✅ {table_name:<35s} - {row_count:>6,d} rows × {col_count:>2d} cols")
        
    except Exception as e:
        print(f"⚠️ {table_name:<35s} - Error: {str(e)[:50]}")

print(f"\n{'='*80}")
print(f"Loaded data for {len(all_data)} tables")
print(f"{'='*80}\n")


Loading 22 sample tables from /Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample



✅ dim_channel                         -      4 rows ×  2 cols
✅ dim_claim_type                      -      6 rows ×  2 cols
✅ dim_member_segment                  -    144 rows ×  8 cols
✅ dim_product_line                    -      5 rows ×  2 cols
✅ dim_providers                       -    539 rows ×  4 cols
✅ dim_region                          -     10 rows ×  3 cols
✅ dm_claims_experience                -  1,000 rows × 23 cols


✅ dm_member_value                     -  1,000 rows × 48 cols
✅ dm_policy_retention                 -     18 rows × 10 cols
✅ dq_monitoring                       -     24 rows ×  8 cols
✅ fact_claims                         -  1,000 rows × 16 cols
✅ fact_members                        -  1,000 rows × 18 cols
✅ fact_policies                       -  1,000 rows × 22 cols
✅ ft_claims_risk                      -  1,000 rows × 18 cols
✅ ft_claims_risk_split                -  1,000 rows × 19 cols
✅ ft_policy_churn                     -  1,000 rows × 23 cols


✅ ft_policy_churn_split               -  1,000 rows × 24 cols
✅ ml_monitoring                       -     29 rows × 11 cols
✅ scored_policy_churn                 -  1,000 rows ×  8 cols
✅ star_claims                         -  1,000 rows × 20 cols
✅ star_members                        -  1,000 rows × 10 cols
✅ star_policies                       -  1,000 rows × 21 cols

Loaded data for 22 tables



In [15]:
# Display sample data from any table
# Change 'table_name' below to view different tables

# List all available tables
print("Available tables:")
for i, table_name in enumerate(sorted(all_data.keys()), 1):
    rows, cols = all_data[table_name]["rows"], all_data[table_name]["cols"]
    print(f"  {i:2d}. {table_name:<35s} ({rows:>6,d} rows × {cols:>2d} cols)")

# Example: View a specific table (change table_name as needed)
example_table = "fact_claims"
if example_table in all_data:
    print(f"\n{'='*80}")
    print(f"Sample data from '{example_table}':")
    print(f"{'='*80}\n")
    print(all_data[example_table]["pandas_df"].head(10))
    print(f"\n{'='*80}")
    print(f"Schema for '{example_table}':")
    print(f"{'='*80}")
    all_data[example_table]["spark_df"].printSchema()

Available tables:
   1. dim_channel                         (     4 rows ×  2 cols)
   2. dim_claim_type                      (     6 rows ×  2 cols)
   3. dim_member_segment                  (   144 rows ×  8 cols)
   4. dim_product_line                    (     5 rows ×  2 cols)
   5. dim_providers                       (   539 rows ×  4 cols)
   6. dim_region                          (    10 rows ×  3 cols)
   7. dm_claims_experience                ( 1,000 rows × 23 cols)
   8. dm_member_value                     ( 1,000 rows × 48 cols)
   9. dm_policy_retention                 (    18 rows × 10 cols)
  10. dq_monitoring                       (    24 rows ×  8 cols)
  11. fact_claims                         ( 1,000 rows × 16 cols)
  12. fact_members                        ( 1,000 rows × 18 cols)
  13. fact_policies                       ( 1,000 rows × 22 cols)
  14. ft_claims_risk                      ( 1,000 rows × 18 cols)
  15. ft_claims_risk_split                ( 1,000 rows × 1

In [ ]:
# Export all tables to CSV, Parquet, or JSON formats
from pathlib import Path

def export_all_tables(format_type="csv", output_base="/Users/manojrammopati/Public/Projects/bupa_insurance_project/data/exports"):
    """
    Export all loaded tables to specified format.
    format_type: 'csv', 'parquet', or 'json'
    """
    output_path = Path(output_base) / format_type
    output_path.mkdir(parents=True, exist_ok=True)
    
    print(f"Exporting {len(all_data)} tables to {format_type.upper()} format...\n")
    
    for table_name, data in all_data.items():
        pdf = data["pandas_df"]
        
        if format_type == "csv":
            file_path = output_path / f"{table_name}.csv"
            pdf.to_csv(file_path, index=False)
        elif format_type == "parquet":
            file_path = output_path / f"{table_name}.parquet"
            pdf.to_parquet(file_path, index=False)
        elif format_type == "json":
            file_path = output_path / f"{table_name}.json"
            pdf.to_json(file_path, orient="records", indent=2)
        else:
            print(f"⚠️ Unknown format: {format_type}")
            return
        
        print(f"✅ {table_name:<35s} → {file_path.name}")
    
    print(f"\n✅ All tables exported to {output_path}")

# Example usage (uncomment to run):
# export_all_tables("csv")
# export_all_tables("parquet")
# export_all_tables("json")